# 01 — Simple RAG (Retrieval-Augmented Generation)

## Overview
The simplest and most foundational RAG pattern:
1. **Index** — Split documents into chunks, embed each chunk, store in a vector DB
2. **Retrieve** — Embed the user query, find the top-K most similar chunks
3. **Generate** — Feed retrieved chunks as context to Claude, get a grounded answer

## Architecture
```
User Query
    │
    ▼
Embed Query  (Amazon Bedrock Titan Embeddings V2)
    │
    ▼
Vector Search  (Qdrant Cloud  ←→  OpenSearch fallback)
    │
    ▼
Top-K Chunks
    │
    ▼
Generate Answer  (Amazon Bedrock Claude Sonnet)
    │
    ▼
Answer + Sources
```

## Vector DB Strategy
| Priority | Backend | Trigger |
|----------|---------|--------|
| 1st | **Qdrant Cloud** | `QDRANT_URL` env var is set |
| 2nd | **OpenSearch Serverless** | `OPENSEARCH_ENDPOINT` env var is set |
| 3rd | **Qdrant In-Memory** | Neither is set (dev/test fallback) |

> Set `QDRANT_URL` and `QDRANT_API_KEY` before running.

## Step 1 — Install & Import Dependencies

In [1]:
# Install required packages (safe to re-run)
import subprocess, sys
packages = [
    "boto3",
    "qdrant-client",
    "opensearch-py",
    "requests-aws4auth",
    "strands-agents",   # AWS Strands — replaces LangChain
    "pypdf",            # Direct PDF reading
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")


All packages ready.


In [2]:
import os, sys, json, time, uuid
from typing import List, Dict, Optional, Tuple

import boto3

# AWS Strands — agent framework used for LLM generation
from strands import Agent
from strands.models.bedrock import BedrockModel

# Qdrant
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, ScoredPoint
)

print("Imports OK")


Imports OK


## Step 2 — Configuration
Edit the values below or set them as environment variables.

In [3]:
# ── AWS / Bedrock ──────────────────────────────────────────────────────────────
AWS_REGION        = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL   = "amazon.titan-embed-text-v2:0"          # 1024-dim
LLM_MODEL         = "us.anthropic.claude-sonnet-4-6"        # Claude Sonnet 4.6

# ── Vector DB ─────────────────────────────────────────────────────────────────
QDRANT_URL        = os.getenv("QDRANT_URL",     "")          # e.g. https://xxx.qdrant.io
QDRANT_API_KEY    = os.getenv("QDRANT_API_KEY", "")          # Qdrant Cloud API key
OPENSEARCH_URL    = os.getenv("OPENSEARCH_ENDPOINT", "")     # OpenSearch endpoint

# ── RAG Parameters ────────────────────────────────────────────────────────────
COLLECTION_NAME   = "simple_rag_01"   # Qdrant collection / OpenSearch index name
EMBEDDING_DIM     = 1024              # Titan V2 output dimension
CHUNK_SIZE        = 1000              # Characters per chunk
CHUNK_OVERLAP     = 200              # Overlap between consecutive chunks
TOP_K             = 5                # Number of chunks to retrieve per query

print(f"Region         : {AWS_REGION}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"LLM model      : {LLM_MODEL}")
print(f"Qdrant URL     : {QDRANT_URL or '(not set — will use in-memory)'}")
print(f"OpenSearch URL : {OPENSEARCH_URL or '(not set)'}")
print(f"Collection     : {COLLECTION_NAME}")

Region         : us-east-1
Embedding model: amazon.titan-embed-text-v2:0
LLM model      : us.anthropic.claude-sonnet-4-6
Qdrant URL     : https://2210ff1c-7c49-4fec-91f4-01662586299c.us-east-1-1.aws.cloud.qdrant.io
OpenSearch URL : (not set)
Collection     : simple_rag_01


## Step 3 — Vector DB Client (Qdrant → OpenSearch fallback)

We define a thin `VectorStore` wrapper that talks Qdrant first and
falls back to OpenSearch Serverless if `QDRANT_URL` is not set.

In [4]:
class VectorStore:
    """
    Unified vector store.
    Priority: Qdrant Cloud → OpenSearch Serverless → Qdrant In-Memory
    
    Public API (same regardless of backend):
      create_collection(dim)           → bool
      upsert(docs)                     → int   docs = [{text, embedding, metadata}]
      search(query_vector, top_k)      → [{text, metadata, score, id}]
      count()                          → int
      delete_collection()              → None
    """

    def __init__(self, collection_name: str, qdrant_url: str = "",
                 qdrant_api_key: str = "", opensearch_url: str = "",
                 region: str = "us-east-1"):

        self.name   = collection_name
        self.region = region
        self._backend = None

        # ── Try Qdrant Cloud ──────────────────────────────────────────────────
        if qdrant_url:
            try:
                kwargs = {"url": qdrant_url}
                if qdrant_api_key:
                    kwargs["api_key"] = qdrant_api_key
                self._qdrant = QdrantClient(**kwargs)
                self._qdrant.get_collections()          # connectivity check
                self._backend = "qdrant_cloud"
                print(f"Connected to Qdrant Cloud: {qdrant_url}")
                return
            except Exception as e:
                print(f"Qdrant Cloud unavailable ({e}), trying next...")

        # ── Try OpenSearch Serverless ─────────────────────────────────────────
        if opensearch_url:
            try:
                from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
                creds = boto3.Session().get_credentials()
                auth  = AWSV4SignerAuth(creds, region, "aoss")
                host  = opensearch_url.replace("https://", "").replace("http://", "")
                self._os = OpenSearch(
                    hosts=[{"host": host, "port": 443}],
                    http_auth=auth, use_ssl=True, verify_certs=True,
                    connection_class=RequestsHttpConnection, timeout=30
                )
                self._os.info()                         # connectivity check
                self._backend = "opensearch"
                print(f"Connected to OpenSearch: {opensearch_url}")
                return
            except Exception as e:
                print(f"OpenSearch unavailable ({e}), falling back to in-memory Qdrant...")

        # ── Fallback: Qdrant In-Memory ────────────────────────────────────────
        self._qdrant  = QdrantClient(":memory:")
        self._backend = "qdrant_memory"
        print("Using Qdrant in-memory (data lost on kernel restart)")

    # ── Create / delete collection ────────────────────────────────────────────

    def create_collection(self, dim: int = 1024, force_recreate: bool = False) -> bool:
        if self._backend in ("qdrant_cloud", "qdrant_memory"):
            exists = self.name in [c.name for c in self._qdrant.get_collections().collections]
            if exists and force_recreate:
                self._qdrant.delete_collection(self.name)
                exists = False
            if not exists:
                self._qdrant.create_collection(
                    self.name,
                    vectors_config=VectorParams(size=dim, distance=Distance.COSINE)
                )
                print(f"Created collection '{self.name}' (dim={dim})")
            else:
                print(f"Collection '{self.name}' already exists")
            return True

        if self._backend == "opensearch":
            if force_recreate and self._os.indices.exists(index=self.name):
                self._os.indices.delete(index=self.name)
            if not self._os.indices.exists(index=self.name):
                body = {
                    "settings": {"index": {"knn": True}},
                    "mappings": {"properties": {
                        "text":      {"type": "text"},
                        "metadata":  {"type": "object"},
                        "embedding": {
                            "type": "knn_vector", "dimension": dim,
                            "method": {"name": "hnsw", "space_type": "cosinesimil",
                                       "engine": "faiss",
                                       "parameters": {"ef_construction": 512, "m": 16}}
                        }
                    }}
                }
                self._os.indices.create(index=self.name, body=body)
                print(f"Created OpenSearch index '{self.name}'")
            else:
                print(f"Index '{self.name}' already exists")
            return True

    # ── Upsert documents ──────────────────────────────────────────────────────

    def upsert(self, docs: List[Dict]) -> int:
        """docs: list of {text, embedding, metadata}"""
        if self._backend in ("qdrant_cloud", "qdrant_memory"):
            points = [
                PointStruct(
                    id=str(uuid.uuid4()),
                    vector=d["embedding"],
                    payload={"text": d["text"], "metadata": d.get("metadata", {})}
                ) for d in docs
            ]
            self._qdrant.upsert(collection_name=self.name, points=points)
            return len(points)

        if self._backend == "opensearch":
            count = 0
            for d in docs:
                self._os.index(index=self.name, body=d)
                count += 1
            time.sleep(1)   # let OpenSearch make docs searchable
            return count

    # ── Vector search ─────────────────────────────────────────────────────────

    def search(self, query_vector: List[float], top_k: int = 5) -> List[Dict]:
        if self._backend in ("qdrant_cloud", "qdrant_memory"):
            resp = self._qdrant.query_points(
                collection_name=self.name,
                query=query_vector,
                limit=top_k,
                with_payload=True
            )
            return [
                {"text": p.payload.get("text", ""),
                 "metadata": p.payload.get("metadata", {}),
                 "score": p.score,
                 "id": str(p.id)}
                for p in resp.points
            ]

        if self._backend == "opensearch":
            body = {
                "size": top_k,
                "query": {"knn": {"embedding": {"vector": query_vector, "k": top_k}}},
                "_source": {"excludes": ["embedding"]}
            }
            resp = self._os.search(index=self.name, body=body)
            return [
                {"text": h["_source"].get("text", ""),
                 "metadata": h["_source"].get("metadata", {}),
                 "score": h["_score"],
                 "id": h["_id"]}
                for h in resp["hits"]["hits"]
            ]

    # ── Utilities ─────────────────────────────────────────────────────────────

    def count(self) -> int:
        if self._backend in ("qdrant_cloud", "qdrant_memory"):
            return self._qdrant.get_collection(self.name).points_count or 0
        if self._backend == "opensearch":
            return self._os.count(index=self.name).get("count", 0)

    def delete_collection(self):
        if self._backend in ("qdrant_cloud", "qdrant_memory"):
            self._qdrant.delete_collection(self.name)
        if self._backend == "opensearch":
            self._os.indices.delete(index=self.name, ignore=[404])
        print(f"Deleted '{self.name}'")

print("VectorStore class defined.")

VectorStore class defined.


## Step 4 — Bedrock Helpers (Embeddings + LLM)

In [5]:
# ── Embeddings: still use boto3 directly (Strands has no embeddings API) ────
bedrock_rt = boto3.client("bedrock-runtime", region_name=AWS_REGION)


def embed_text(text: str) -> List[float]:
    """1024-dim Titan V2 embedding for a single string."""
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json"
    )
    return json.loads(resp["body"].read())["embedding"]


def embed_batch(texts: List[str]) -> List[List[float]]:
    """Embed a list of texts; Titan V2 processes one at a time."""
    embeddings = []
    for i, t in enumerate(texts):
        embeddings.append(embed_text(t))
        if (i + 1) % 10 == 0:
            print(f"  Embedded {i+1}/{len(texts)}")
        time.sleep(0.05)
    return embeddings


# ── Generation: use AWS Strands BedrockModel + Agent ─────────────────────────
_strands_model = BedrockModel(
    model_id=LLM_MODEL,
    region_name=AWS_REGION
)


def generate_answer(question: str, context_chunks: List[str]) -> str:
    """
    Generate a grounded answer with AWS Strands Agent.
    The agent receives retrieved chunks as context and answers using only them.
    """
    context = "\n\n".join(
        f"[Chunk {i+1}]\n{chunk}" for i, chunk in enumerate(context_chunks)
    )
    prompt = (
        f"Use ONLY the context below to answer the question. "
        f"If the answer is not in the context, say 'Not found in context.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\nAnswer:"
    )
    agent = Agent(
        model=_strands_model,
        system_prompt="You are a precise Q&A assistant. Answer only from the provided context."
    )
    result = agent(prompt)
    return str(result)


# Quick smoke test for embeddings
test_emb = embed_text("Hello world")
print(f"Embedding OK  — dim={len(test_emb)}, sample={[round(x, 4) for x in test_emb[:3]]}")
print("Strands Agent configured — BedrockModel ready")


Embedding OK  — dim=1024, sample=[-0.0326, 0.0692, 0.0018]
Strands Agent configured — BedrockModel ready


## Step 5 — Connect to Vector Store & Create Collection

In [6]:
# Initialise the vector store — tries Qdrant Cloud first, then OpenSearch, then in-memory
vs = VectorStore(
    collection_name=COLLECTION_NAME,
    qdrant_url=QDRANT_URL,
    qdrant_api_key=QDRANT_API_KEY,
    opensearch_url=OPENSEARCH_URL,
    region=AWS_REGION
)
print(f"\nActive backend: {vs._backend}")

# Create the collection (force_recreate=True wipes any old data from a previous run)
vs.create_collection(dim=EMBEDDING_DIM, force_recreate=True)

Connected to Qdrant Cloud: https://2210ff1c-7c49-4fec-91f4-01662586299c.us-east-1-1.aws.cloud.qdrant.io

Active backend: qdrant_cloud
Created collection 'simple_rag_01' (dim=1024)


True

## Step 6 — Load & Chunk the PDF

We load `data/climate.pdf` (a 13-page Advanced Climatology paper) using **`pypdf`** directly
(no LangChain dependency), then split into overlapping chunks with a native Python
recursive character splitter — the same algorithm used by LangChain, re-implemented
without any third-party dependency.

**Reuse pattern:** all subsequent notebooks use the same PDF — just change `PDF_PATH`.

| Parameter | Value | Why |
|-----------|-------|-----|
| `CHUNK_SIZE` | 1000 chars | Fits comfortably in Claude's context |
| `CHUNK_OVERLAP` | 200 chars | Preserves context across chunk boundaries |
| Metadata | page_num + source | Enables citation in answers |


In [7]:
import os
import pypdf

# ── Native recursive character splitter (no LangChain) ─────────────────────
def recursive_split(text: str, chunk_size: int = 1000,
                    chunk_overlap: int = 200) -> "List[str]":
    """Split text into overlapping chunks: paragraph, line, word, char."""
    separators = ["\n\n", "\n", ". ", " ", ""]

    def _split(text, seps):
        sep = ""
        new_seps = []
        for i, s in enumerate(seps):
            if s == "" or s in text:
                sep = s
                new_seps = seps[i+1:]
                break

        parts = text.split(sep) if sep != "" else list(text)

        good = []
        for part in parts:
            if len(part) <= chunk_size:
                good.append(part)
            elif new_seps:
                good.extend(_split(part, new_seps))
            else:
                for k in range(0, len(part), chunk_size - chunk_overlap):
                    good.append(part[k:k+chunk_size])

        chunks, cur_pieces, cur_len = [], [], 0
        for piece in good:
            p = piece.strip()
            if not p:
                continue
            addition = len(sep) + len(p) if cur_pieces else len(p)
            if cur_len + addition <= chunk_size:
                cur_pieces.append(p)
                cur_len += addition
            else:
                if cur_pieces:
                    chunks.append(sep.join(cur_pieces))
                overlap_pieces, overlap_len = [], 0
                for prev in reversed(cur_pieces):
                    if overlap_len + len(prev) + len(sep) <= chunk_overlap:
                        overlap_pieces.insert(0, prev)
                        overlap_len += len(prev) + len(sep)
                    else:
                        break
                cur_pieces = overlap_pieces + [p]
                cur_len = sum(len(x) + len(sep) for x in cur_pieces)
        if cur_pieces:
            chunks.append(sep.join(cur_pieces))
        return [c for c in chunks if c.strip()]

    return _split(text, separators)


# ── Load PDF with pypdf ────────────────────────────────────────────────────
PDF_PATH = os.path.join("..", "data", "climate.pdf")

reader = pypdf.PdfReader(PDF_PATH)
print(f"PDF loaded   : {PDF_PATH}")
print(f"Total pages  : {len(reader.pages)}")
print("\nPage 1 preview (first 400 chars):")
print(reader.pages[0].extract_text()[:400])

# ── Chunk every page ──────────────────────────────────────────────────────
chunks = []
for page_num, page in enumerate(reader.pages):
    page_text = page.extract_text() or ""
    for chunk_text in recursive_split(page_text, CHUNK_SIZE, CHUNK_OVERLAP):
        if chunk_text.strip():
            chunks.append({
                "text"    : chunk_text.strip(),
                "page_num": page_num + 1,
                "source"  : os.path.basename(PDF_PATH)
            })

print(f"\nChunks created   : {len(chunks)}")
print(f"Avg chunk length : {sum(len(c['text']) for c in chunks) / len(chunks):.0f} chars")
print("\nSample chunk (chunk 0):")
print(chunks[0]["text"][:300])


PDF loaded   : ..\data\climate.pdf
Total pages  : 13

Page 1 preview (first 400 chars):
 
 
 
 
 
 
TOPIC: - 
WEATHER ANALYSIS AND WEATHERFORECASTING. 
 
 
 
 
 
 
PAPER NAME: - ADVANCED CLIMATOLOGY 
SUBJECT: - GEOGRAPHY  
SEMESTER: - M.A –II 
PAPER CODE: - (GEO-201) 
UNIVERSITY DEPARTMENT OF GEOGRAPHY, 
DR. SHYMA PRASAD MUKHERJEE UNIVERSITY, RANCHI. 
 
 

Chunks created   : 38
Avg chunk length : 847 chars

Sample chunk (chunk 0):
TOPIC: -
WEATHER ANALYSIS AND WEATHERFORECASTING.
PAPER NAME: - ADVANCED CLIMATOLOGY
SUBJECT: - GEOGRAPHY
SEMESTER: - M.A –II
PAPER CODE: - (GEO-201)
UNIVERSITY DEPARTMENT OF GEOGRAPHY,
DR. SHYMA PRASAD MUKHERJEE UNIVERSITY, RANCHI.


## Step 7 — Generate Embeddings and Index

Each chunk is embedded with Titan V2 (1024 dims) and stored in the vector store.

In [8]:
print(f"Generating embeddings for {len(chunks)} chunks...")
t0 = time.time()

texts      = [c["text"] for c in chunks]
embeddings = embed_batch(texts)

# Build index records — metadata carries page number for citation
docs_to_index = [
    {
        "text"     : chunks[i]["text"],
        "embedding": embeddings[i],
        "metadata" : {
            "chunk_index": i,
            "page_num"   : chunks[i]["page_num"],
            "source"     : chunks[i]["source"]
        }
    }
    for i in range(len(chunks))
]

indexed = vs.upsert(docs_to_index)
elapsed = time.time() - t0

print(f"\nIndexed  : {indexed} chunks")
print(f"Time     : {elapsed:.2f}s  ({elapsed/len(chunks):.2f}s per chunk)")
print(f"Count in Qdrant collection: {vs.count()}")


Generating embeddings for 38 chunks...
  Embedded 10/38
  Embedded 20/38
  Embedded 30/38

Indexed  : 38 chunks
Time     : 6.81s  (0.18s per chunk)
Count in Qdrant collection: 38


## Step 8 — Query the RAG System

For each test question:
1. Embed the question with Titan
2. Retrieve top-K chunks from the vector store
3. Pass chunks + question to Claude → get answer

In [9]:
def rag_query(question: str, top_k: int = TOP_K, verbose: bool = True) -> Dict:
    """
    Full RAG pipeline for one question.
    Steps: embed question -> vector search -> format context -> Claude generates answer.
    Returns dict with question, answer, sources, latency_ms.
    """
    t0 = time.time()

    # A: embed the question with Titan V2
    q_embedding = embed_text(question)

    # B: retrieve top-K most similar chunks from Qdrant
    results = vs.search(q_embedding, top_k=top_k)

    # C: generate answer — Claude sees the retrieved chunks as context
    context_texts = [r["text"] for r in results]
    answer = generate_answer(question, context_texts)

    latency_ms = (time.time() - t0) * 1000

    if verbose:
        print(f"\nQuestion : {question}")
        print(f"Latency  : {latency_ms:.0f}ms")
        print(f"\nAnswer:\n{answer}")
        print(f"\nTop sources (with page citations):")
        for i, r in enumerate(results[:3], 1):
            page = r['metadata'].get('page_num', '?')
            print(f"  [{i}] page={page}  score={r['score']:.4f}  {r['text'][:100]}...")
        print("-" * 70)

    return {"question": question, "answer": answer,
            "sources": results, "latency_ms": latency_ms}


# Climate / weather forecasting questions matched to climate.pdf content
test_questions = [
    "What is weather forecasting and why is it important?",
    "What are the main methods used in weather analysis?",
    "How does climatology differ from meteorology?",
    "What factors influence weather patterns and climate?",
]

results_log = []
for q in test_questions:
    r = rag_query(q)
    results_log.append(r)


## Weather Forecasting: Definition and Importance

### Definition
According to the context, **weather forecasting** is defined as the **prediction of atmospheric conditions** — including air temperature, humidity, sky conditions, air pressure, and general circulation of the atmosphere — of a particular place or region, using **scientific tools and technological knowledge**. It is supplemented by a variety of statistical and empirical techniques and can be done at different temporal levels (daily, weekly, monthly, etc.).

---

### Why It Is Important

Weather forecasting is important across many sectors of society:

- **Tourism** – Tourists planning trips require forecasts to plan activities and take necessary precautions in advance.
- **Fishing** – Fishermen need forecasts ranging from a few hours to several days, depending on the duration of their fishing trips.
- **Sports** – The success of outdoor games and tournaments depends on favourable weather conditions.
- **Agriculture** – It

## Step 9 — Evaluation & Metrics

In [10]:
# Keyword-coverage evaluation: did the answer contain expected domain terms?
eval_cases = [
    {"question": "What is weather forecasting and why is it important?",
     "expected_keywords": ["forecast", "weather", "predict", "atmosphere", "climate"]},
    {"question": "What are the main methods used in weather analysis?",
     "expected_keywords": ["analysis", "synoptic", "observation", "data", "pressure"]},
    {"question": "How does climatology differ from meteorology?",
     "expected_keywords": ["climate", "weather", "long", "study", "atmosphere"]},
]

print(f"{'Question':<55} {'KW Hit':>7} {'Latency':>10}")
print("-" * 75)

for case in eval_cases:
    result = rag_query(case["question"], verbose=False)
    answer_lower = result["answer"].lower()
    hits  = sum(1 for kw in case["expected_keywords"] if kw.lower() in answer_lower)
    total = len(case["expected_keywords"])
    print(f"{case['question'][:54]:<55} {hits}/{total} ({hits/total*100:.0f}%) {result['latency_ms']:>8.0f}ms")

print()
avg_lat = sum(r["latency_ms"] for r in results_log) / len(results_log)
print(f"Average latency across all queries: {avg_lat:.0f}ms")


Question                                                 KW Hit    Latency
---------------------------------------------------------------------------
## Weather Forecasting: Definition and Importance

### Definition
According to the context, **weather forecasting** is the *"prediction of atmospheric conditions like air temperature, humidity, sky conditions, air pressure and general circulation of the atmosphere of a particular place or a region using scientific tools and technological knowledge."* It predicts atmospheric conditions before they actually happen and is supplemented by statistical and empirical techniques.

---

### Importance

Weather forecasting is important across many sectors of society:

- **Tourism** – Tourists planning trips require complete weather forecasts to plan accordingly and take necessary precautions in advance.
- **Fishing** – Fishermen need forecasts ranging from a few hours to several days, depending on how long they go out to sea.
- **Sports** – The su

## Step 10 — Summary

### What we built
| Component | Implementation |
|-----------|---------------|
| **PDF Loading** | `pypdf.PdfReader` (direct, no LangChain) |
| **Chunking** | Native Python recursive splitter (1000 chars, 200 overlap) |
| **Embeddings** | Amazon Bedrock Titan V2 — 1024 dims |
| **Vector DB** | Qdrant Cloud (falls back to OpenSearch → in-memory) |
| **LLM** | AWS Strands `Agent` + Bedrock Claude Sonnet 4.6 |
| **Retrieval** | Cosine similarity, top-5 |

### Strands framework role
- `BedrockModel` wraps the Bedrock runtime config (model ID, region, credentials)
- `Agent` manages the generation call — system prompt, message routing, response extraction
- Embeddings still use boto3 directly (Strands has no embeddings API)

### Strengths
- No LangChain dependency — pure AWS stack (Strands + Bedrock + Qdrant)
- Same recursive chunking algorithm, implemented natively
- Good baseline for comparison with advanced patterns

### Limitations
- No reranking — top-K may include low-quality matches
- Fixed chunk size — may cut across important context boundaries
- Single query — no query expansion or reformulation

### Next: **02 — Graph RAG** (entities + relationships, multi-hop queries)


In [ ]:
# Optional cleanup — uncomment to delete the Qdrant collection after testing
# vs.delete_collection()

print(f"Collection '{COLLECTION_NAME}' retained in {vs._backend}.")
print(f"Total vectors stored: {vs.count()}")
print("\nDone. Give the go-ahead to proceed to notebook 02.")